Dans ce notebook, nous allons construire notre premier modèle de recommandation basé sur le contenu des films. 

Le modèle recommande des films similaires à un film donné, en se basant uniquement sur les genres, sans regarder du tout qui a aimé quoi. C'est différent du filtrage collaboratif (prochain notebook), qui lui se base sur les comportements des utilisateurs.

Une fonction qui répond à la question : "Si un utilisateur a aimé le film X, quels autres films pourrait-il aimer ?"

Exemple concret : on va donner "Toy Story" en entrée et la fonction nous ressort une liste de films avec des genres similaires (autres films d'animation/famille/comédie), classés du plus similaire au moins similaire.

Dans MovieLens, la seule information de "contenu" facilement disponible est la colonne "genres". Le content-based ne se base que sur les genres, on pourrait l'enrichir avec les tags ou des métadonnées TMDb comme le synopsis".



Voici les grands étapes de ce notebook : 

- Transformer le texte en nombres : un ordinateur ne comprend pas "Comedy", il faut convertir les genres en vecteurs numériques. C'est le rôle du TF-IDF

- Mesurer la ressemblance entre films : une fois les films transformés en vecteurs, on calcule à quel point deux vecteurs se ressemblent → c'est le rôle de la similarité cosinus

- Construire une fonction utilisable : à partir d'un titre de film, retrouver les films les plus proches selon cette similarité

In [35]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

movies = pd.read_csv('/Users/nazmanazirhussain/Desktop/RecommandationFilm/data/nettoye/movies_clean.csv')
movies['genres_list'] = movies['genres'].str.split('|')

print(movies.shape)
movies.head()

(9742, 6)


,movieId,title,genres,annee,titre,genres_list
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995,Toy Story,"[Adventure, Animation, Children, Comedy, Fantasy]"
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995,Jumanji,"[Adventure, Children, Fantasy]"
2,3,Grumpier Old Men (1995),Comedy|Romance,1995,Grumpier Old Men,"[Comedy, Romance]"
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995,Waiting to Exhale,"[Comedy, Drama, Romance]"
4,5,Father of the Bride Part II (1995),Comedy,1995,Father of the Bride Part II,[Comedy]


In [36]:
movies['genresSep'] = movies['genres'].fillna('').str.replace('|', ' ')
movies[['title', 'genres', 'genresSep']].head()

,title,genres,genresSep
0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Adventure Animation Children Comedy Fantasy
1,Jumanji (1995),Adventure|Children|Fantasy,Adventure Children Fantasy
2,Grumpier Old Men (1995),Comedy|Romance,Comedy Romance
3,Waiting to Exhale (1995),Comedy|Drama|Romance,Comedy Drama Romance
4,Father of the Bride Part II (1995),Comedy,Comedy


Construction de la matrice TF-IDF (Term Frequency - Inverse Document Frequency) 

In [37]:
# On indique qu'un "mot" est une suite de caractères sans espace.
# Ça évite que "Sci-Fi" ou "Film-Noir" soient coupés en deux à cause du tiret.
tfidf = TfidfVectorizer(token_pattern=r'[^\s]+')
tfidf_matrix = tfidf.fit_transform(movies['genresSep'])

print("Dimensions de la matrice TF-IDF :", tfidf_matrix.shape)
print("Vocabulaire (genres) :", tfidf.get_feature_names_out())

Dimensions de la matrice TF-IDF : (9742, 19)
Vocabulaire (genres) : ['action' 'adventure' 'animation' 'children' 'comedy' 'crime'
 'documentary' 'drama' 'fantasy' 'film-noir' 'horror' 'imax' 'musical'
 'mystery' 'romance' 'sci-fi' 'thriller' 'war' 'western']


Calcul de la similarité entre tous les films

In [38]:
# On compare chaque film à tous les autres films selon leurs vecteurs de genres.
# Résultat : une matrice carrée où [i, j] = score de similarité entre film i et film j (entre 0 et 1).
meme_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)

print("Dimensions de la matrice de similarité :", meme_matrix.shape)

Dimensions de la matrice de similarité : (9742, 9742)


Fonction de recommandation 

In [39]:
def filmSimilaire(titre_film, n=10):
    # Trouver la position du film demandé (recherche sur 'titre', sans l'année)
    idx = movies[movies['titre'] == titre_film].index[0]
    
    # Récupérer sa similarité avec tous les autres films
    scores = list(enumerate(meme_matrix[idx]))
    
    # Trier du plus similaire au moins similaire (on exclut le film lui-même avec [1:n+1])
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:n+1]
    
    # Récupérer les titres correspondants
    indices = [i[0] for i in scores]
    return movies.iloc[indices][['titre', 'genres', 'annee']]

On teste si la fonction marche ou pas : La fonction fonctionne très bien. On peut observer que tous les films proposés ont exactement les mêmes genres que Toy Story. Il y a donc un score de similarité de 1.0 chacun. C'est cohérent. Le modèle content-based basé sur les genres seuls atteint ses limites lorsque plusieurs films partagent une combinaison de genres identique, produisant des scores de similarité maximale (1.0) sans possibilité de différenciation plus fine. Cette limite pourrait être réduite en enrichissant le vecteur avec les tags ou le synopsis des films.

In [40]:
filmSimilaire('Toy Story')

,titre,genres,annee
1706,Antz,Adventure|Animation|Children|Comedy|Fantasy,1998
2355,Toy Story 2,Adventure|Animation|Children|Comedy|Fantasy,1999
2809,"Adventures of Rocky and Bullwinkle, The",Adventure|Animation|Children|Comedy|Fantasy,2000
3000,"Emperor's New Groove, The",Adventure|Animation|Children|Comedy|Fantasy,2000
3568,"Monsters, Inc.",Adventure|Animation|Children|Comedy|Fantasy,2001
6194,"Wild, The",Adventure|Animation|Children|Comedy|Fantasy,2006
6486,Shrek the Third,Adventure|Animation|Children|Comedy|Fantasy,2007
6948,"Tale of Despereaux, The",Adventure|Animation|Children|Comedy|Fantasy,2008
7760,Asterix and the Vikings (Astérix et les Vikings),Adventure|Animation|Children|Comedy|Fantasy,2006
8219,Turbo,Adventure|Animation|Children|Comedy|Fantasy,2013


Pour valider que le modèle marche aussi dans des cas plus variés, on va faire plusieurs tests. 

In [41]:
filmSimilaire('Pulp Fiction')

,titre,genres,annee
520,Fargo,Comedy|Crime|Drama|Thriller,1996
791,Freeway,Comedy|Crime|Drama|Thriller,1996
2453,Man Bites Dog (C'est arrivé près de chez vous),Comedy|Crime|Drama|Thriller,1992
3155,Beautiful Creatures,Comedy|Crime|Drama|Thriller,2000
4169,Confessions of a Dangerous Mind,Comedy|Crime|Drama|Thriller,2002
4523,Party Monster,Comedy|Crime|Drama|Thriller,2003
6676,In Bruges,Comedy|Crime|Drama|Thriller,2008
7129,"Informant!, The",Comedy|Crime|Drama|Thriller,2009
7293,Leaves of Grass,Comedy|Crime|Drama|Thriller,2009
20,Get Shorty,Comedy|Crime|Thriller,1995


In [42]:
filmSimilaire('Matrix, The')

,titre,genres,annee
68,Screamers,Action|Sci-Fi|Thriller,1995
144,Johnny Mnemonic,Action|Sci-Fi|Thriller,1995
296,Virtuosity,Action|Sci-Fi|Thriller,1995
336,Timecop,Action|Sci-Fi|Thriller,1994
474,Blade Runner,Action|Sci-Fi|Thriller,1982
567,Solo,Action|Sci-Fi|Thriller,1996
601,"Arrival, The",Action|Sci-Fi|Thriller,1996
939,"Terminator, The",Action|Sci-Fi|Thriller,1984
1373,Godzilla,Action|Sci-Fi|Thriller,1998
1939,"Matrix, The",Action|Sci-Fi|Thriller,1999


On peut constater que le modèle capture bien ce type de recommandation même si ce n'est que par les genres. 

Code de sauvegarde du modèle

In [43]:
import pickle

with open('/Users/nazmanazirhussain/Desktop/RecommandationFilm/models/matrice_tfidf.pkl', 'wb') as fichier:
    pickle.dump(tfidf_matrix, fichier)

with open('/Users/nazmanazirhussain/Desktop/RecommandationFilm/models/matrice_similarite.pkl', 'wb') as fichier:
    pickle.dump(meme_matrix, fichier)

with open('/Users/nazmanazirhussain/Desktop/RecommandationFilm/models/vectoriseur_tfidf.pkl', 'wb') as fichier:
    pickle.dump(tfidf, fichier)